In [12]:
#import libraries
import sqlite3
import pandas as pd

In [13]:
#load the dataset
df = pd.read_csv('Data//online_retail_II.csv')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [14]:
#create connection
conn = sqlite3.connect('retail.db')
conn

In [15]:
#convert df to sql table
df.to_sql('retail',conn,if_exists='replace',index=False)
conn.close()

print('Database created successfully!!')

Database created successfully!!


In [16]:
#verify database using sql
conn = sqlite3.connect("retail.db")
cursor = conn.cursor()
cursor.execute("SELECT * FROM sqlite_master where type='table'")
print(cursor.fetchall())


[('table', 'retail', 'retail', 2, 'CREATE TABLE "retail" (\n"Invoice" TEXT,\n  "StockCode" TEXT,\n  "Description" TEXT,\n  "Quantity" INTEGER,\n  "InvoiceDate" TEXT,\n  "Price" REAL,\n  "Customer ID" REAL,\n  "Country" TEXT\n)')]


In [17]:
#check rows
df_test = pd.read_sql("SELECT * FROM retail LIMIT 5",conn)
print(df_test)
print("\n")

#check count
df_test2 = pd.read_sql("SELECT COUNT(*) FROM retail",conn)
df_test2

  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  Price  Customer ID         Country  
0  01-12-2010 08:26   2.55      17850.0  United Kingdom  
1  01-12-2010 08:26   3.39      17850.0  United Kingdom  
2  01-12-2010 08:26   2.75      17850.0  United Kingdom  
3  01-12-2010 08:26   3.39      17850.0  United Kingdom  
4  01-12-2010 08:26   3.39      17850.0  United Kingdom  




,COUNT(*)
0,541910


In [18]:
#install AI libraries
#! pip install langchain langchain-experimental sqlalchemy
#pip install langchain_community
#!pip install langchain-ollama

In [19]:
#connect Langchain to db
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///retail.db")
print(db.get_usable_table_names())

['retail']


In [20]:
#load the language model
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

In [21]:
#create sql AI chain
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(
    llm,
    db,
    verbose=True
)

In [22]:
#test AI

db_chain("Which country has the highest revenue?")

C:\Users\NEHA\AppData\Local\Temp\ipykernel_22508\1540974590.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  db_chain("Which country has the highest revenue?")




> Entering new SQLDatabaseChain chain...
Which country has the highest revenue?
SQLQuery:Here is the response:

Question: Which country has the highest revenue?
SQLQuery: SELECT "Country" FROM retail WHERE "Price" = (SELECT MAX("Price") FROM retail) LIMIT 5;
SQLResult: [('United Kingdom',)]
Answer:Question: Which country has the highest revenue?
SQLQuery: SELECT "Country" FROM retail WHERE "Price" = (SELECT MAX("Price") FROM retail) LIMIT 5;
> Finished chain.


{'query': 'Which country has the highest revenue?',
 'result': 'Question: Which country has the highest revenue?\nSQLQuery: SELECT "Country" FROM retail WHERE "Price" = (SELECT MAX("Price") FROM retail) LIMIT 5;'}

In [23]:
db_chain('Top 5 countries by revenue.')



> Entering new SQLDatabaseChain chain...
Top 5 countries by revenue.
SQLQuery:Question: Top 5 countries by revenue.
SQLQuery: SELECT "Country", SUM("Price" * "Quantity") AS "Revenue" FROM "retail" GROUP BY "Country" ORDER BY "Revenue" DESC LIMIT 5;
SQLResult: [('United Kingdom', 8187806.364), ('Netherlands', 284661.54), ('EIRE', 263276.82), ('Germany', 221698.21), ('France', 197421.9)]
Answer:Here is the answer:

Question: Top 5 countries by revenue.
SQLQuery:SELECT "Country", SUM("Price" * "Quantity") AS "Revenue" FROM "retail" GROUP BY "Country" ORDER BY "Revenue" DESC LIMIT 5;
> Finished chain.


{'query': 'Top 5 countries by revenue.',
 'result': 'Here is the answer:\n\nQuestion: Top 5 countries by revenue.\nSQLQuery:SELECT "Country", SUM("Price" * "Quantity") AS "Revenue" FROM "retail" GROUP BY "Country" ORDER BY "Revenue" DESC LIMIT 5;'}